# Archive historique — evaluer_lot_pilote

Ce notebook est conservé pour documenter la préparation et les évaluations antérieures. Il ne doit pas être exécuté pour régénérer les jeux de test déjà validés.

In [ ]:
"""Évalue les prédictions existantes sur les documents validés du lot pilote.

Usage : python3 Scripts/pipeline_et_validation/archive_mise_au_point/evaluer_lot_pilote.ipynb

La détection d'une annonce est appariée une à une par (document, nom normalisé).
Lorsqu'un même nom apparaît plusieurs fois dans un document, la fonction la plus
similaire départage les candidats. Les champs sont ensuite évalués séparément.
"""

from __future__ import annotations

import csv
import difflib
import re
import unicodedata
from collections import defaultdict
from pathlib import Path


def trouver_racine() -> Path:
    courant = Path.cwd().resolve()
    for candidat in (courant, *courant.parents):
        if (candidat / "total_donnees").is_dir() and (candidat / "evaluation_extraction").is_dir():
            return candidat
    raise FileNotFoundError("Ouvre ce notebook depuis le dossier de travail.")

RACINE = trouver_racine()
DOSSIER = RACINE / "evaluation_extraction"
CHAMPS = ("date_conseil", "civilite", "matricule", "fonction", "ministere")


def lire(nom: str) -> list[dict[str, str]]:
    with (DOSSIER / nom).open(encoding="utf-8-sig", newline="") as fichier:
        return list(csv.DictReader(fichier))


def ecrire_csv(nom: str, colonnes: list[str], lignes: list[dict[str, str]]) -> None:
    with (DOSSIER / nom).open("w", encoding="utf-8-sig", newline="") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=colonnes, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(lignes)


def normaliser(valeur: str) -> str:
    sans_accents = "".join(
        caractere
        for caractere in unicodedata.normalize("NFKD", (valeur or "").casefold())
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^a-z0-9]", "", sans_accents)


def f1(precision: float, rappel: float) -> float:
    return 0.0 if precision + rappel == 0 else 2 * precision * rappel / (precision + rappel)


def apparier(predictions: list[dict[str, str]], references: list[dict[str, str]]):
    """Retourne les paires (index prédiction, index référence) et les restes."""
    references_libres = set(range(len(references)))
    paires = []
    faux_positifs = []
    for index_prediction, prediction in enumerate(predictions):
        candidats = [
            index_reference
            for index_reference in references_libres
            if prediction["document_id"] == references[index_reference]["document_id"]
            and normaliser(prediction["nom"]) == normaliser(references[index_reference]["nom"])
        ]
        if not candidats:
            faux_positifs.append(index_prediction)
            continue

        # Les homonymes ou doubles nominations portent généralement des fonctions
        # distinctes. Le score ne crée jamais de rapprochement entre deux noms.
        def score(index_reference: int) -> tuple[float, bool, bool]:
            reference = references[index_reference]
            return (
                difflib.SequenceMatcher(
                    None,
                    normaliser(prediction["fonction"]),
                    normaliser(reference["fonction"]),
                ).ratio(),
                bool(normaliser(prediction["matricule"]))
                and normaliser(prediction["matricule"]) == normaliser(reference["matricule"]),
                bool(normaliser(prediction["ministere"]))
                and normaliser(prediction["ministere"]) == normaliser(reference["ministere"]),
            )

        index_reference = max(candidats, key=score)
        references_libres.remove(index_reference)
        paires.append((index_prediction, index_reference, score(index_reference)[0]))
    return paires, faux_positifs, sorted(references_libres)


def main() -> None:
    lot = lire("lot_pilote_12_documents.csv")
    documents = [ligne["document_id"] for ligne in lot]
    references = [
        ligne
        for ligne in lire("reference_annotations_validees.csv")
        if ligne["document_id"] in documents
    ]
    predictions = [
        ligne
        for ligne in lire("predictions_a_verifier.csv")
        if ligne["document_id"] in documents
    ]
    paires, faux_positifs, faux_negatifs = apparier(predictions, references)

    resultats = []
    par_document: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
    champs_corrects = defaultdict(int)
    champs_comparables = defaultdict(int)
    stricts = 0

    for index_prediction, index_reference, similarite in paires:
        prediction, reference = predictions[index_prediction], references[index_reference]
        strict = all(
            normaliser(prediction[champ]) == normaliser(reference[champ])
            for champ in ("date_conseil", "nom", "fonction")
        )
        stricts += strict
        ligne = {
            "resultat": "vrai_positif_personne",
            "similarite_fonction": f"{similarite:.3f}",
            "fonction_exacte": str(normaliser(prediction["fonction"]) == normaliser(reference["fonction"])).lower(),
            "evenement_strict": str(strict).lower(),
            "document_id": prediction["document_id"],
            "ligne_csv_prediction": prediction["ligne_csv"],
            "nom_prediction": prediction["nom"],
            "fonction_prediction": prediction["fonction"],
            "nom_reference": reference["nom"],
            "fonction_reference": reference["fonction"],
            "page_reference": reference["page"],
        }
        for champ in CHAMPS:
            comparable = bool(normaliser(prediction[champ])) and bool(normaliser(reference[champ]))
            exact = comparable and normaliser(prediction[champ]) == normaliser(reference[champ])
            ligne[f"{champ}_exact"] = str(exact).lower()
            champs_comparables[champ] += comparable
            champs_corrects[champ] += exact
        resultats.append(ligne)
        par_document[prediction["document_id"]]["vp"] += 1
        par_document[prediction["document_id"]]["strict"] += strict

    for index_prediction in faux_positifs:
        prediction = predictions[index_prediction]
        resultats.append({
            "resultat": "faux_positif_personne",
            "document_id": prediction["document_id"],
            "ligne_csv_prediction": prediction["ligne_csv"],
            "nom_prediction": prediction["nom"],
            "fonction_prediction": prediction["fonction"],
        })
        par_document[prediction["document_id"]]["fp"] += 1

    for index_reference in faux_negatifs:
        reference = references[index_reference]
        resultats.append({
            "resultat": "faux_negatif_personne",
            "document_id": reference["document_id"],
            "nom_reference": reference["nom"],
            "fonction_reference": reference["fonction"],
            "page_reference": reference["page"],
        })
        par_document[reference["document_id"]]["fn"] += 1

    colonnes = [
        "resultat", "similarite_fonction", "fonction_exacte", "evenement_strict",
        "document_id", "ligne_csv_prediction", "nom_prediction", "fonction_prediction",
        "nom_reference", "fonction_reference", "page_reference",
        *[f"{champ}_exact" for champ in CHAMPS],
    ]
    ecrire_csv("resultats_appariement_lot_pilote.csv", colonnes, resultats)

    vp, fp, fn = len(paires), len(faux_positifs), len(faux_negatifs)
    precision, rappel = vp / (vp + fp), vp / (vp + fn)
    precision_stricte = stricts / len(predictions)
    rappel_strict = stricts / len(references)
    lignes_tableau = []
    for document in documents:
        compte = par_document[document]
        doc_vp, doc_fp, doc_fn = compte["vp"], compte["fp"], compte["fn"]
        doc_pred = doc_vp + doc_fp
        doc_ref = doc_vp + doc_fn
        doc_precision = doc_vp / doc_pred if doc_pred else None
        doc_rappel = doc_vp / doc_ref if doc_ref else None
        # Sans prédiction mais avec une référence, le F1 est nul. Lorsque le
        # document ne contient aucune annonce ni prédiction, il reste indéfini.
        doc_f1 = (
            f1(doc_precision or 0.0, doc_rappel or 0.0)
            if doc_pred or doc_ref
            else None
        )
        lignes_tableau.append(
            f"| {document} | {doc_pred} | {doc_ref} | {doc_vp} | {doc_fp} | {doc_fn} | "
            f"{compte['strict']} | "
            f"{'—' if doc_precision is None else f'{doc_precision:.1%}'} | "
            f"{'—' if doc_rappel is None else f'{doc_rappel:.1%}'} | "
            f"{'—' if doc_f1 is None else f'{doc_f1:.1%}'} |"
        )
    lignes_champs = []
    for champ in CHAMPS:
        taux = champs_corrects[champ] / champs_comparables[champ] if champs_comparables[champ] else 0
        lignes_champs.append(f"| `{champ}` | {champs_corrects[champ]} | {champs_comparables[champ]} | {taux:.1%} |")

    rapport = f"""# Évaluation exploratoire de l’extracteur — lot pilote

*Calculée le 27 septembre 2026 à partir des 12 PDF relus intégralement.*

Le lot comporte **566 annonces de référence** dans onze documents positifs et un document négatif traité par OCR. L’extracteur contient **375 prédictions** pour ces documents.

## Détection des personnes nommées

Une prédiction est appariée une à une à une référence lorsque le document et le nom normalisé correspondent. Quand la même personne apparaît plusieurs fois dans un document, la fonction la plus semblable départage les annonces. Cette mesure indique si l’extracteur retrouve la personne dans le bon compte rendu ; les champs sont contrôlés séparément.

| VP | FP | FN | Précision | Rappel | F1 |
|---:|---:|---:|---:|---:|---:|
| {vp} | {fp} | {fn} | {precision:.1%} | {rappel:.1%} | {f1(precision, rappel):.1%} |

L’extracteur produit peu de personnes hors référence ({fp} faux positifs), mais omet {fn} annonces. Les faux positifs incluent les signatures du Président du Faso ou du Premier ministre reprises comme si elles étaient des nominations.

## Événement avec fonction exacte

Une annonce est considérée exacte sur le noyau `document + date + nom + fonction` seulement si ces quatre éléments concordent après normalisation des accents, de la casse et de la ponctuation.

| Annonces exactes | Précision | Rappel | F1 |
|---:|---:|---:|---:|
| {stricts} | {precision_stricte:.1%} | {rappel_strict:.1%} | {f1(precision_stricte, rappel_strict):.1%} |

## Exactitude des champs parmi les personnes appariées

Un champ est comparé seulement lorsqu’il est renseigné à la fois dans la prédiction et la référence. Le dénominateur ne mesure donc pas la couverture d’un champ vide.

| Champ | Correct | Comparable | Exactitude |
|---|---:|---:|---:|
{chr(10).join(lignes_champs)}

## Résultats par document

| Document | Prédictions | Références | VP | FP | FN | Événements exacts | Précision | Rappel | F1 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
{chr(10).join(lignes_tableau)}

Le document négatif du 16 janvier 2016 ne comporte ni prédiction ni référence : une précision et un F1 ne sont donc pas définis pour lui. Le document du 9 juin 2021 contient 36 nominations de conseils d’administration et aucune prédiction : il montre une lacune importante de couverture.

## Limites d’interprétation

Ce lot sert à la mise au point et à l’évaluation exploratoire : des documents ont été examinés pendant la conception de l’annotation assistée. Ces résultats ne sont pas un test final indépendant. Réserver un autre échantillon, non consulté pendant les ajustements, avant de publier une performance finale.

Les détails de chaque appariement, faux positif et faux négatif sont dans [`resultats_appariement_lot_pilote.csv`](resultats_appariement_lot_pilote.csv). Ils permettent d’examiner les erreurs de fonction et de compléter une typologie d’erreurs dans le mémoire.
"""
    (DOSSIER / "RAPPORT_EVALUATION_LOT_PILOTE.md").write_text(rapport, encoding="utf-8")
    print(f"VP={vp}, FP={fp}, FN={fn}, F1={f1(precision, rappel):.3f}; "
          f"événements stricts={stricts}; rapport créé.")


## Exécution manuelle

Vérifier les chemins et les fichiers de sortie avant toute exécution.

In [ ]:
# main()